# Training a small transformer on enwik8

**GPU**: `Runtime > Change runtime type > T4 GPU`

Trains a small byte-level GPT (no tokenizer, same ASCII-level setup as Delétang et al.) on a few MB of enwik8, then extracts per-byte loss on a fresh held-out chunk - the input for the geometry/Fourier analysis in `research_project_v3.ipynb`, Section 9.

Run `MAX_STEPS = 100` first as a sanity check, then set to `3000` (or more) for the real run.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('Warning: GPU is not enabled! Go to Runtime > Change runtime type and enable a GPU.')
else:
    print('GPU:', torch.cuda.get_device_name(0))
torch.manual_seed(11)

device: cuda
GPU: Tesla T4


## Data
Upload `enwik8_first_55MB.txt`.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving enwik8_first_55MB.txt to enwik8_first_55MB.txt


In [3]:
uploaded ="/content/enwik8_first_55MB.txt"

In [4]:
with open('enwik8_first_55MB.txt', 'rb') as f:
    raw = f.read()
print('total bytes:', len(raw))

data = np.frombuffer(raw, dtype=np.uint8).astype(np.int64)

TRAIN_BYTES = 5_000_000
train_data = torch.tensor(data[:TRAIN_BYTES], dtype=torch.long)
heldout_data = torch.tensor(data[TRAIN_BYTES:TRAIN_BYTES + 600_000], dtype=torch.long)
print('train:', len(train_data), ' heldout (completely fresh, unseen by the model):', len(heldout_data))

total bytes: 57671680
train: 5000000  heldout (completely fresh, unseen by the model): 600000


## Architecture
4 layers, 4 heads, embedding dim 128, context 256. ~1M parameters.

In [5]:
VOCAB_SIZE = 256
BLOCK_SIZE = 256
N_LAYER = 4
N_HEAD = 4
N_EMBD = 128
DROPOUT = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(N_EMBD, 3*N_EMBD)
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.n_head = N_HEAD
        self.drop = nn.Dropout(DROPOUT)
        mask = torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)).view(1, 1, BLOCK_SIZE, BLOCK_SIZE)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(N_EMBD, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (hd ** 0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(N_EMBD)
        self.mlp = nn.Sequential(
            nn.Linear(N_EMBD, 4*N_EMBD), nn.GELU(),
            nn.Linear(4*N_EMBD, N_EMBD), nn.Dropout(DROPOUT)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb = nn.Parameter(torch.zeros(1, BLOCK_SIZE, N_EMBD))
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:, :T, :]
        x = self.drop(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), targets.view(-1), reduction='mean')
        return logits, loss

model = TinyGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print('number of parameters:', n_params)

number of parameters: 891648


## Training
`MAX_STEPS = 100` for a quick check, `3000` for a real run.

In [6]:
def get_batch(data_split, batch_size):
    ix = torch.randint(0, len(data_split) - BLOCK_SIZE - 1, (batch_size,))
    x = torch.stack([data_split[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data_split[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

BATCH_SIZE = 64
MAX_STEPS = 7000
LR = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
t0 = time.time()
model.train()
for step in range(MAX_STEPS):
    xb, yb = get_batch(train_data, BATCH_SIZE)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 200 == 0 or step == MAX_STEPS - 1:
        print(f'step {step:5d}  loss {loss.item():.4f}  bits/byte {loss.item()/0.6931:.4f}  ({time.time()-t0:.0f}s)')

print('training finished. total time:', round(time.time()-t0), 'seconds')

step     0  loss 5.7063  bits/byte 8.2330  (1s)
step   200  loss 2.7109  bits/byte 3.9112  (16s)
step   400  loss 2.6360  bits/byte 3.8032  (31s)
step   600  loss 2.5610  bits/byte 3.6950  (46s)
step   800  loss 2.4714  bits/byte 3.5657  (62s)
step  1000  loss 2.4494  bits/byte 3.5340  (79s)
step  1200  loss 2.4396  bits/byte 3.5198  (97s)
step  1400  loss 2.3325  bits/byte 3.3653  (116s)
step  1600  loss 2.2237  bits/byte 3.2084  (134s)
step  1800  loss 2.1631  bits/byte 3.1210  (151s)
step  2000  loss 2.0722  bits/byte 2.9897  (169s)
step  2200  loss 2.0629  bits/byte 2.9763  (187s)
step  2400  loss 1.9847  bits/byte 2.8635  (206s)
step  2600  loss 1.9427  bits/byte 2.8030  (223s)
step  2800  loss 1.9375  bits/byte 2.7954  (241s)
step  3000  loss 1.8753  bits/byte 2.7057  (259s)
step  3200  loss 1.8120  bits/byte 2.6144  (277s)
step  3400  loss 1.8205  bits/byte 2.6267  (295s)
step  3600  loss 1.8371  bits/byte 2.6506  (313s)
step  3800  loss 1.7479  bits/byte 2.5218  (331s)
step  40

### Resuming from a checkpoint
If Colab disconnects, run this before the training cell (after the model is built):
```python
from google.colab import files
uploaded = files.upload()  # checkpoint_stepXXXX.pt
model.load_state_dict(torch.load('checkpoint_stepXXXX.pt'))
```

In [7]:
model.eval()
n_chunks = len(heldout_data) // BLOCK_SIZE
per_position_bits = np.zeros(n_chunks * (BLOCK_SIZE - 1), dtype=np.float32)

with torch.no_grad():
    for c in range(n_chunks):
        chunk = heldout_data[c*BLOCK_SIZE:(c+1)*BLOCK_SIZE]
        x = chunk[:-1].unsqueeze(0).to(device)
        y = chunk[1:].unsqueeze(0).to(device)
        logits, _ = model(x)
        logp = F.log_softmax(logits, dim=-1)
        token_logp = logp.gather(-1, y.unsqueeze(-1)).squeeze(-1).squeeze(0)
        bits = (-token_logp / np.log(2)).cpu().numpy()

        start = c*(BLOCK_SIZE-1)
        per_position_bits[start:start+(BLOCK_SIZE-1)] = bits
        if c % 100 == 0:
            print('chunk', c, '/', n_chunks)

print('mean bits per byte (on never-before-seen data):', per_position_bits.mean())
print('every', BLOCK_SIZE-1, 'values correspond to one separate context-limited chunk (non-overlapping, no synthetic data)')

chunk 0 / 2343
chunk 100 / 2343
chunk 200 / 2343
chunk 300 / 2343
chunk 400 / 2343
chunk 500 / 2343
chunk 600 / 2343
chunk 700 / 2343
chunk 800 / 2343
chunk 900 / 2343
chunk 1000 / 2343
chunk 1100 / 2343
chunk 1200 / 2343
chunk 1300 / 2343
chunk 1400 / 2343
chunk 1500 / 2343
chunk 1600 / 2343
chunk 1700 / 2343
chunk 1800 / 2343
chunk 1900 / 2343
chunk 2000 / 2343
chunk 2100 / 2343
chunk 2200 / 2343
chunk 2300 / 2343
mean bits per byte (on never-before-seen data): 2.2755852
every 255 values correspond to one separate context-limited chunk (non-overlapping, no synthetic data)


## Extracting per-byte loss

Non-overlapping context-length chunks on held-out data, following the paper's Section 3.1 setup. Saves `neural_lm_losses.npy` and `neural_lm_meta.npy`.

In [8]:
np.save('neural_lm_losses.npy', per_position_bits)
np.save('neural_lm_meta.npy', np.array([BLOCK_SIZE, n_chunks, N_LAYER, N_HEAD, N_EMBD, n_params]))

from google.colab import files
files.download('neural_lm_losses.npy')
files.download('neural_lm_meta.npy')
print('ready')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ready


## Extra validation: shuffled control + matched-range comparison

Two follow-up checks, run right here while `model`, `heldout_data`, and the full `data` array are still in memory. Neither number is guaranteed in advance -- this just reports what comes out.

In [9]:
def spectrum_zscore(cost_signal, target_period):
    sig = cost_signal - cost_signal.mean()
    spec = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(len(sig), d=1.0)
    idx = np.argmin(np.abs(freqs - 1.0/target_period))
    others = np.delete(spec[1:], idx-1)
    z = (spec[idx]-others.mean())/others.std()
    return z

def extract_losses(model, heldout_tensor):
    model.eval()
    n_chunks = len(heldout_tensor) // BLOCK_SIZE
    bits = np.zeros(n_chunks * (BLOCK_SIZE - 1), dtype=np.float32)
    with torch.no_grad():
        for c in range(n_chunks):
            chunk = heldout_tensor[c*BLOCK_SIZE:(c+1)*BLOCK_SIZE]
            x = chunk[:-1].unsqueeze(0).to(device)
            y = chunk[1:].unsqueeze(0).to(device)
            logits, _ = model(x)
            logp = F.log_softmax(logits, dim=-1)
            token_logp = logp.gather(-1, y.unsqueeze(-1)).squeeze(-1).squeeze(0)
            bits[c*(BLOCK_SIZE-1):(c+1)*(BLOCK_SIZE-1)] = (-token_logp / np.log(2)).cpu().numpy()
    return bits

# --- shuffled control ---
real_losses = per_position_bits  # already computed above
z_real = spectrum_zscore(real_losses, BLOCK_SIZE - 1)

heldout_shuf = heldout_data[torch.randperm(len(heldout_data))]
shuf_losses = extract_losses(model, heldout_shuf)
z_shuf = spectrum_zscore(shuf_losses, BLOCK_SIZE - 1)

print(f'real held-out:     mean={real_losses.mean():.4f} bits/byte, context-reset z={z_real:.2f}')
print(f'shuffled control:  mean={shuf_losses.mean():.4f} bits/byte, context-reset z={z_shuf:.2f}')

real held-out:     mean=2.2756 bits/byte, context-reset z=20.77
shuffled control:  mean=9.1079 bits/byte, context-reset z=21.50


In [10]:
# --- matched-range comparison vs. the large model ---
# bytes [55,000,000 : 57,671,680] are untouched by both this model's and the large model's
# train/heldout ranges (see additional_validation.ipynb, Section 4)
MATCH_START, MATCH_END = 55_000_000, 57_671_680
matched_bytes = torch.tensor(data[MATCH_START:MATCH_END], dtype=torch.long)
matched_losses = extract_losses(model, matched_bytes)
print(f'small model on the matched range [{MATCH_START:,}:{MATCH_END:,}]: {matched_losses.mean():.4f} bits/byte')

np.save('small_model_matched_losses.npy', matched_losses)
from google.colab import files
files.download('small_model_matched_losses.npy')
print('saved -- run the matching cell in train_neural_lm_v2.ipynb, then compare the two means directly')

small model on the matched range [55,000,000:57,671,680]: 2.3291 bits/byte


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

saved -- run the matching cell in train_neural_lm_v2.ipynb, then compare the two means directly
